In [1]:
import time
import random
import pandas as pd

from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager


districts = [
    "Balod",
    "Baloda Bazar",
    "Balrampur",
    "Bastar",
    "Bemetara",
    "Bijapur",
    "Bilaspur",
    "Dantewada",
    "Dhamtari",
    "Durg",
    "Gariaband",
    "Gaurela-Pendra-Marwahi",
    "Janjgir-Champa",
    "Jashpur",
    "Kabirdham",
    "Kanker",
    "Khairagarh-Chhuikhadan-Gandai",
    "Kondagaon",
    "Korba",
    "Korea",
    "Mahasamund",
    "Manendragarh-Chirmiri-Bharatpur",
    "Mohla-Manpur-Ambagarh Chowki",
    "Mungeli",
    "Narayanpur",
    "Raigarh",
    "Raipur",
    "Rajnandgaon",
    "Sakti",
    "Sarangarh-Bilaigarh",
    "Sukma",
    "Surajpur",
    "Surguja"
]


search_types = [
    "hospitals"
]


options = webdriver.ChromeOptions()

options.add_argument("--start-maximized")

options.add_argument(
    "--disable-blink-features=AutomationControlled"
)

options.add_argument("--disable-notifications")


driver = webdriver.Chrome(

    service=Service(
        ChromeDriverManager().install()
    ),

    options=options
)


driver.get("https://www.google.com/maps")

print("Opening Google Maps...")

time.sleep(15)

print("Google Maps Loaded")


all_data = []

visited_links = set()


def get_search_box():

    selectors = [

        '//input[@id="searchboxinput"]',
        '//input[contains(@placeholder,"Search")]',
        '//input[contains(@aria-label,"Search")]',
        '//input'

    ]

    for selector in selectors:

        try:

            box = driver.find_element(
                By.XPATH,
                selector
            )

            return box

        except:
            pass

    return None


def extract_school_data():

    try:

        try:
            name = driver.find_element(
                By.TAG_NAME,
                "h1"
            ).text
        except:
            name = ""

        try:
            address = driver.find_element(
                By.XPATH,
                '//button[contains(@data-item-id,"address")]'
            ).text
        except:
            address = ""

        try:
            phone = driver.find_element(
                By.XPATH,
                '//button[contains(@data-item-id,"phone")]'
            ).text
        except:
            phone = ""

        try:
            website = driver.find_element(
                By.XPATH,
                '//a[contains(@data-item-id,"authority")]'
            ).get_attribute("href")
        except:
            website = ""

        # RATING
        try:
            rating = driver.find_element(
                By.XPATH,
                '//div[@role="img"]'
            ).get_attribute("aria-label")
        except:
            rating = ""

        return {

            "Hospital Name": name,
            "Address": address,
            "Phone": phone,
            "Website": website,
            "Rating": rating,
            "Google Maps URL": driver.current_url

        }

    except:

        return None


for district in districts:

    for search_type in search_types:

        try:

            query = (
                f"{search_type} "
                f"in {district} Chhattisgarh"
            )

            print(f"\nSearching: {query}")


            search_box = get_search_box()

            if not search_box:

                print("Search Box Not Found")

                continue

            ActionChains(driver).move_to_element(
                search_box
            ).click().perform()

            time.sleep(1)

            search_box.send_keys(
                Keys.CONTROL + "a"
            )

            time.sleep(1)

            search_box.send_keys(Keys.DELETE)

            time.sleep(1)

            # TYPE SLOWLY
            for char in query:

                search_box.send_keys(char)

                time.sleep(0.03)

            search_box.send_keys(Keys.ENTER)

            print("Search Submitted")

            time.sleep(8)


            try:

                scrollable_div = driver.find_element(
                    By.XPATH,
                    '//div[@role="feed"]'
                )

                previous_count = 0

                stagnant = 0

                for i in range(100):

                    driver.execute_script(
                        """
                        arguments[0].scrollTop =
                        arguments[0].scrollHeight
                        """,
                        scrollable_div
                    )

                    # SLOWER WAIT
                    time.sleep(4)

                    listings = driver.find_elements(
                        By.XPATH,
                        '//a[contains(@href,"/maps/place/")]'
                    )

                    current_count = len(listings)

                    print(
                        f"{district} | "
                        f"{search_type} | "
                        f"Scroll {i+1} -> "
                        f"{current_count}"
                    )

                    if current_count == previous_count:

                        stagnant += 1

                    else:

                        stagnant = 0

                    # STOP AFTER 4 SAME COUNTS
                    if stagnant >= 4:

                        break

                    previous_count = current_count

            except Exception as e:

                print(f"Scrolling Error: {e}")


            listings = driver.find_elements(
                By.XPATH,
                '//a[contains(@href,"/maps/place/")]'
            )

            links = []

            for listing in listings:

                try:

                    link = listing.get_attribute("href")

                    if (
                        link
                        and link not in visited_links
                    ):

                        links.append(link)

                        visited_links.add(link)

                except:
                    pass

            print(f"Collected {len(links)} links")


            for index, link in enumerate(links):

                try:

                    print(
                        f"{district} | "
                        f"{index+1}/{len(links)}"
                    )

                    driver.get(link)

                    time.sleep(
                        random.uniform(2, 4)
                    )

                    data = extract_school_data()

                    if data:

                        data["District"] = district

                        data["Search Type"] = search_type

                        all_data.append(data)

                        print(data["Hospital Name"])


                    if len(all_data) % 100 == 0:

                        backup_df = pd.DataFrame(
                            all_data
                        )

                        backup_df.drop_duplicates(
                            subset=[
                                "Hospital Name",
                                "Google Maps URL"
                            ],
                            inplace=True
                        )

                        backup_filename = (
                            "chhattisgarh_backup.csv"
                        )

                        backup_df.to_csv(
                            backup_filename,
                            index=False
                        )

                        print("Backup Saved")

                except Exception as e:

                    print(
                        f"School Error: {e}"
                    )

        except Exception as e:

            print(
                f"District Error: {e}"
            )


df = pd.DataFrame(all_data)

df.drop_duplicates(

    subset=[
        "Hospital Name",
        "Google Maps URL"
    ],

    inplace=True
)

filename = (
    "chhattisgarh_hospitals.csv"
)

df.to_csv(
    filename,
    index=False
)

print("\nExtraction Completed")

print(
    f"Total Schools Extracted: {len(df)}"
)

print(f"\nSaved File: {filename}")

driver.quit()


Opening Google Maps...
Google Maps Loaded

Searching: hospitals in Balod Chhattisgarh
Search Submitted
Balod | hospitals | Scroll 1 -> 10
Balod | hospitals | Scroll 2 -> 10
Balod | hospitals | Scroll 3 -> 10
Balod | hospitals | Scroll 4 -> 10
Balod | hospitals | Scroll 5 -> 10
Collected 10 links
Balod | 1/10
Umeed Health Care Multispeciality Hospital
Balod | 2/10
Shree Narsinh Multispeciality Hospital
Balod | 3/10
Sagar Hospital
Balod | 4/10
District Hospital Balod
Balod | 5/10
Shri Prakash Chand Memorial Hospital
Balod | 6/10
Kilkari Nursing Home
Balod | 7/10
Dist Hospital Balod
Balod | 8/10
SANJEEVANI Hospital
Balod | 9/10
Govt.Hospital Suregaon
Balod | 10/10
Government Ayurvedic Hospital

Searching: hospitals in Baloda Bazar Chhattisgarh
Search Submitted
Baloda Bazar | hospitals | Scroll 1 -> 14
Baloda Bazar | hospitals | Scroll 2 -> 20
Baloda Bazar | hospitals | Scroll 3 -> 20
Baloda Bazar | hospitals | Scroll 4 -> 20
Baloda Bazar | hospitals | Scroll 5 -> 20
Baloda Bazar | hospita